# Tumour identification pipeline

Four stages, each driven by one cell:
1. **Discover jobs** — scan a folder, build `manifest.csv`.
2. **Auto-detect tumour** — run brightfield (and optional fluorescence) segmentation, save a 4-channel curation TIF per job.
3. **Curate labels** *(optional)* — open napari to fix BF/FL masks.
4. **Quantify** — combine curated/auto masks into a single `tumour_areas.csv`.

Output layout (flat, one TIF per job; pixel size is embedded in the TIF resolution tags):
```
<OUTPUT_DIR>/manifest.csv
<OUTPUT_DIR>/stage2/<name>.tif    # auto: 4 ch uint8 [BF_raw, FL_raw|0, BF_label, FL_label|0]
<OUTPUT_DIR>/stage3/<name>.tif    # curated overrides (same layout) — only after Stage 3
<OUTPUT_DIR>/tumour_areas.csv     # written by Stage 4
```


In [1]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))
from tumour_id import discover_jobs, auto_detect, label_curation, quantify

# ── Pipeline parameters (edit as needed) ─────────────────────────────────────
SOURCE_DIR = Path(r"Z:\Bel\Marina_Side_Projects\Area_PDO\For_Analysis\Inputs")
OUTPUT_DIR = Path(r"Z:\Bel\Marina_Side_Projects\Area_PDO\For_Analysis\Outputs")
# Channels are auto-inferred per image: BF = channel 0, FL = last channel
# (only when the image has >1 channel).
# ─────────────────────────────────────────────────────────────────────────────

MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

## Stage 1 — Discover jobs

In [2]:
discover_jobs.build_and_write_manifest(
    source_dir=SOURCE_DIR,
    output_dir=OUTPUT_DIR,
    manifest_path=MANIFEST_PATH,
)

Wrote manifest: Z:\Bel\Marina_Side_Projects\Area_PDO\For_Analysis\Outputs\manifest.csv  (116/116 to process)


WindowsPath('Z:/Bel/Marina_Side_Projects/Area_PDO/For_Analysis/Outputs/manifest.csv')

## Stage 2 — Auto-detect tumour

Produces a 4-channel `stage2/<name>.tif` per job (BF raw, FL raw, BF mask, FL mask) with the pixel size embedded in the TIF resolution tags. Per-image failures are printed but never abort the batch.


In [3]:
auto_detect.run(MANIFEST_PATH)

Stage 2: processing 116 / 116 jobs from Z:\Bel\Marina_Side_Projects\Area_PDO\For_Analysis\Outputs\manifest.csv
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL12_device1_img0
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL12_device2_img1
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL12_device3_img2
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL12_device4_img3
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL12_device1_img4
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL12_device2_img5
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL12_device3_img6
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL12_device4_img7
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL32_device1_img8
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL32_device2_img9
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL32_device3_img10
  [OK  ] 2026.05.14_Size_CART_batchPerm_UTD_FL32_device4_img11
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL32_device1_img12
  [OK  ] 2026.05.14_Size_CART_batchPerm_ARi_FL32_device2_img13
  [OK  ] 2026.05.

## Stage 3 — Curate labels *(optional)*

Opens a napari viewer that loops over every job with a `stage2/<name>.tif`. Edit the BF / FL label layers, use **Previous** / **Next** to navigate (your edits to the current image are saved when you move away), then click **Done — Write Overrides** to write `stage3/<name>.tif` for every job (use the auto mask as-is by simply not editing it).

Skip this cell entirely if you want to use the auto-detected masks for everything.


In [2]:
label_curation.main([str(MANIFEST_PATH)])

Opening curation viewer for 116 job(s)...


0

## Stage 4 — Quantify

Prefers `stage3/<name>.tif` (curated) and falls back to `stage2/<name>.tif` (auto). Pixel size is read from the TIF resolution tags. Writes a single summary CSV with BF and (when present) FL areas in pixels and µm².


In [ ]:
quantify.run(MANIFEST_PATH, csv_path=OUTPUT_DIR / "tumour_areas.csv")